In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load datasets
train = pd.read_csv("../data/train.csv")
calendar = pd.read_csv("../data/calendar_events.csv")
submission_template = pd.read_csv("../data/forecast_submission.csv")

LOOKBACK_DAYS = 28

In [25]:
from sklearn.linear_model import LinearRegression

TREND_DAYS = 84   # try 56 (or 84). 28 is often too noisy.
trend_per_store = {}

train["date"] = pd.to_datetime(train["date"])

for store_id, df in train.groupby("store_id"):
    df = df.sort_values("date").tail(TREND_DAYS)

    X = np.arange(len(df)).reshape(-1, 1)
    y = df["revenue"].values

    model = LinearRegression()
    model.fit(X, y)

    trend_per_store[store_id] = model.coef_[0]  # daily slope


In [26]:
list(trend_per_store.items())[:5]


[(0, np.float64(-116.70680267287645)),
 (1, np.float64(-18.403000000000016)),
 (2, np.float64(28.79139181937836)),
 (3, np.float64(-30.51853194289765)),
 (4, np.float64(7.5053256049407615))]

In [27]:
# Start from the weekday-based submission logic
sub = submission_template.copy()
sub["store_id"] = sub["id"].str.split("_").str[0].astype(int)
sub["date_str"] = sub["id"].str.split("_").str[1]
sub["date"] = pd.to_datetime(sub["date_str"], format="%Y%m%d")
sub["weekday"] = sub["date"].dt.weekday


In [28]:
train["date"] = pd.to_datetime(train["date"])
last_date = train["date"].max()
sub["days_ahead"] = (sub["date"] - last_date).dt.days


In [29]:
# make sure train date is datetime
train["date"] = pd.to_datetime(train["date"])

LOOKBACK_DAYS = 28

recent = (
    train.sort_values("date")
         .groupby("store_id")
         .tail(LOOKBACK_DAYS)
         .copy()
)

recent["weekday"] = recent["date"].dt.weekday

store_weekday_mean = recent.groupby(["store_id", "weekday"])["revenue"].mean()
store_mean = recent.groupby("store_id")["revenue"].mean()

store_weekday_mean.head(), store_mean.head()


(store_id  weekday
 0         0          291526.8000
           1          258842.7875
           2          254890.9200
           3          264703.6125
           4          293333.5425
 Name: revenue, dtype: float64,
 store_id
 0    298830.560714
 1     35568.902143
 2     32421.249643
 3     49842.347500
 4     19691.897143
 Name: revenue, dtype: float64)

In [30]:
# --- Base predictors ---
sub["p_weekday"] = sub.set_index(["store_id", "weekday"]).index.map(store_weekday_mean)
sub["p_store"]   = sub["store_id"].map(store_mean)

# Fallback if weekday mean missing
sub["p_weekday"] = sub["p_weekday"].fillna(sub["p_store"])

# --- Blend weekday + store mean ---
ALPHA = 0.9 
sub["prediction"] = ALPHA * sub["p_weekday"] + (1 - ALPHA) * sub["p_store"]

# --- Trend (map slope) ---
sub["trend"] = sub["store_id"].map(trend_per_store).fillna(0.0)

# --- Trend safety rails ---
# 1) cap trend relative to store scale (e.g., max ±1% of store_mean per day)
CAP_FRAC = 0.005
cap = sub["p_store"] * CAP_FRAC
sub["trend_capped"] = sub["trend"].clip(lower=-cap, upper=cap)

# 2) only apply trend for first HORIZON_CLIP days
HORIZON_CLIP = 7
sub["days_used"] = sub["days_ahead"].clip(lower=0, upper=HORIZON_CLIP)

# 3) damping (shrink the trend effect)
DAMP = 0.1 
sub["prediction"] = sub["prediction"] + DAMP * sub["trend_capped"] * sub["days_used"]


In [31]:
sub[["id", "prediction"]].head()

,id,prediction
0,0_20151001,268104.636641
1,0_20151002,293859.902961
2,0_20151003,355352.191531
3,0_20151004,359996.862100
4,0_20151005,292198.822670


In [32]:
final_submission = sub[["id", "prediction"]].copy()

print("Rows:", len(final_submission))
print("Any NaNs?", final_submission["prediction"].isna().any())

final_submission.to_csv("../submissions/blend_trend_capped_submission.csv", index=False)
print("Saved blend_trend_capped_submission.csv")


Rows: 1012
Any NaNs? False
Saved blend_trend_capped_submission.csv
